# Perceptrón desde cero
Implementación con `numpy` y `matplotlib`. Solo se usa `sklearn` para generar datos.

## 1. Imports y Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

np.random.seed(42)

X, y = make_blobs(n_samples=100, centers=2, cluster_std=1.2, random_state=42)
y = np.where(y == 0, -1, 1)

plt.figure(figsize=(6, 4))
plt.scatter(X[y==-1,0], X[y==-1,1], c='steelblue', label='Clase -1', edgecolors='k')
plt.scatter(X[y== 1,0], X[y== 1,1], c='tomato',    label='Clase +1', edgecolors='k')
plt.title('Dataset generado'); plt.legend(); plt.tight_layout(); plt.show()

## 2. Clase Perceptrón

In [ ]:
class Perceptron:
    def __init__(self, n_features, lr=0.1):
        self.w  = np.random.randn(n_features) * 0.01
        self.b  = 0.0
        self.lr = lr
        self.errors_per_epoch = []

    def activation(self, z):
        """Funcion escalon: +1 si z >= 0, -1 si z < 0."""
        return np.where(z >= 0, 1, -1)

    def predict(self, X):
        """Forward pass: X @ w + b -> activacion."""
        return self.activation(X @ self.w + self.b)

    def train(self, X, y, epochs=50):
        """Regla del perceptron: w += lr * (y - y_hat) * x"""
        for _ in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_hat = self.predict(xi.reshape(1, -1))[0]
                if yi != y_hat:
                    delta   = self.lr * (yi - y_hat)
                    self.w += delta * xi
                    self.b += delta
                    errors += 1
            self.errors_per_epoch.append(errors)
            if errors == 0:
                break

## 3. Entrenamiento y frontera de decision

In [ ]:
def decision_boundary(perceptron, x_range):
    w = perceptron.w
    if abs(w[1]) < 1e-9:
        return None, None
    return x_range, -(w[0] * x_range + perceptron.b) / w[1]

perc = Perceptron(n_features=2, lr=0.1)
x_range = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200)

xb_antes, yb_antes = decision_boundary(perc, x_range)
perc.train(X, y, epochs=100)
xb_despues, yb_despues = decision_boundary(perc, x_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, xb, yb, titulo in [
    (axes[0], xb_antes,   yb_antes,   'Antes de entrenar'),
    (axes[1], xb_despues, yb_despues, 'Despues de entrenar'),
]:
    ax.scatter(X[y==-1,0], X[y==-1,1], c='steelblue', edgecolors='k', label='Clase -1')
    ax.scatter(X[y== 1,0], X[y== 1,1], c='tomato',    edgecolors='k', label='Clase +1')
    if xb is not None:
        ax.plot(xb, yb, 'k--', lw=2, label='Frontera')
    ax.set_title(titulo); ax.legend()
plt.tight_layout(); plt.show()

acc = np.mean(perc.predict(X) == y)
print(f'Accuracy final: {acc*100:.1f}%')
print(f'Epocas hasta converger: {len(perc.errors_per_epoch)}')

## 4. Evolucion del error por epoca

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(perc.errors_per_epoch, marker='o', color='darkorange')
plt.xlabel('Epoca'); plt.ylabel('Errores'); plt.title('Errores por epoca')
plt.tight_layout(); plt.show()

## 5. Datos NO linealmente separables

In [ ]:
X_nl, y_nl = make_blobs(n_samples=150, centers=2, cluster_std=2.5, random_state=7)
y_nl = np.where(y_nl == 0, -1, 1)

perc_nl = Perceptron(n_features=2, lr=0.1)
perc_nl.train(X_nl, y_nl, epochs=100)

xr = np.linspace(X_nl[:,0].min()-1, X_nl[:,0].max()+1, 200)
_, yb = decision_boundary(perc_nl, xr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_nl[y_nl==-1,0], X_nl[y_nl==-1,1], c='steelblue', edgecolors='k')
axes[0].scatter(X_nl[y_nl== 1,0], X_nl[y_nl== 1,1], c='tomato',    edgecolors='k')
axes[0].plot(xr, yb, 'k--', lw=2)
axes[0].set_title('Datos solapados - frontera final')
axes[1].plot(perc_nl.errors_per_epoch, color='purple')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Errores')
axes[1].set_title('Error NO converge a 0')
plt.tight_layout(); plt.show()
print('El error nunca llega a 0 porque no existe hiperplano que separe perfectamente las clases.')

## 6. Por que el perceptron no puede resolver XOR?

El perceptron encuentra una **linea recta** que separa dos clases.
XOR no es linealmente separable: no existe ninguna recta que divida correctamente sus 4 puntos.

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([-1, 1, 1, -1])

perc_xor = Perceptron(n_features=2, lr=0.1)
perc_xor.train(X_xor, y_xor, epochs=100)

xx, yy = np.meshgrid(np.linspace(-0.5,1.5,300), np.linspace(-0.5,1.5,300))
Z = perc_xor.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='bwr')
plt.scatter(X_xor[y_xor==-1,0], X_xor[y_xor==-1,1], c='steelblue', s=200, edgecolors='k', zorder=5, label='XOR=0')
plt.scatter(X_xor[y_xor== 1,0], X_xor[y_xor== 1,1], c='tomato',    s=200, edgecolors='k', zorder=5, label='XOR=1')
plt.title('XOR - el perceptron NO puede separarlo linealmente')
plt.legend(); plt.tight_layout(); plt.show()

acc_xor = np.mean(perc_xor.predict(X_xor) == y_xor)
print(f'Accuracy en XOR: {acc_xor*100:.0f}%  (maximo posible con perceptron simple: 75%)')

## 7. Efecto del learning rate

In [ ]:
lrs = [0.001, 0.1, 1.0]
results = {}
for lr in lrs:
    p = Perceptron(n_features=2, lr=lr)
    p.train(X, y, epochs=200)
    results[lr] = p.errors_per_epoch

plt.figure(figsize=(8, 4))
for (lr, errors), color in zip(results.items(), ['royalblue','darkorange','green']):
    plt.plot(errors, label=f'lr={lr}', color=color)
plt.xlabel('Epoca'); plt.ylabel('Errores')
plt.title('Comparacion de learning rates')
plt.legend(); plt.tight_layout(); plt.show()

for lr, errors in results.items():
    print(f'lr={lr:.3f} -> epocas: {len(errors)}, errores finales: {errors[-1]}')

**Analisis:**
- **lr=0.001:** converge muy lento, necesita muchas epocas.
- **lr=0.1:** balance entre velocidad y estabilidad. Generalmente el mejor.
- **lr=1.0:** converge rapido si los datos son separables, puede oscilar con ruido.

## 8. Cuantas epocas necesito para converger?

El **Teorema de Convergencia del Perceptron** garantiza convergencia en pasos finitos **solo si** los datos son linealmente separables.

Depende de:
- **Separabilidad** (margen entre clases)
- **Learning rate**
- **Tamano y ruido** del dataset
- **Inicializacion aleatoria** de pesos

Si los datos **no son separables**, el perceptron oscila indefinidamente.

In [ ]:
print(f'Epocas hasta converger (lr=0.1, datos separables): {len(perc.errors_per_epoch)}')